# STAT163 · Practice 5 — tidy data from two publishers

One notebook across your two practice sessions. Submit the changes in it.

Both tables here are files as their publishers put them on the web: the World Bank and
Eurostat, the statistical office of the European Union. A table is **tidy** when each
variable is a column and each observation is a row. Neither of these two is. You make each
one tidy with `melt`, and with `pivot_table` you build wide tables for reading.

Each part has **Task** cells with a self-check under them. They are the required minimum,
and they carry the grade: a Task cell earns its marks by holding an answer that runs. Each
part ends with an **Explore and discuss** block, which is optional. The note in Part 3 and
the disclosure cell get feedback.

**AI in this practice: recommended mode.**

Try each task yourself first.

If you are stuck in the classroom, ask the teacher or TA. Working remotely: ask your LLM to
explain the concept rather than to solve. Two boundaries are firm:

- say what you used, in the disclosure cell at the end
- do not submit a notebook authored end to end by AI. It is not your work and it is not
  accepted.

## Before you start

Replace the placeholder with your real name and run the cell.

In [ ]:
from selfcheck import check, check_col, check_identity

student_name = "Name Surname"   # ← replace with your name

check_identity(student_name)

### `melt` and `pivot_table` on a small table

Below are the medals that three countries won at the Paris 2024 Olympic Games: one row per
country and one column per kind of medal. The notebook `01-tidy-data.ipynb` of the `week5-pre-lecture`
repository has more worked examples.

In [ ]:
import pandas as pd

medals = pd.DataFrame({
    "country": ["United States", "China", "Japan"],
    "gold": [40, 40, 20],
    "silver": [44, 27, 12],
    "bronze": [42, 24, 13],
})
medals

With `melt` you make a table longer: the column headers become the values of one column. In
`id_vars` you name the columns that stay as they are, in `var_name` the new column for the
old headers, and in `value_name` the new column for the values.

In [ ]:
medals_long = medals.melt(id_vars="country", var_name="medal", value_name="count")
medals_long

One row is now one country and one kind of medal.

With `pivot_table` you do the opposite: the values of one column become column headers. In
`index` you name the columns that identify one row of the result, in `columns` the column
whose values become the new headers, and in `values` the column that fills them.

In [ ]:
medals_long.pivot_table(index="country", columns="medal", values="count")

The new columns and the rows come out sorted by name, and the counts as decimals. Each cell
holds a mean: here the mean of one row. A mean is a decimal number, so 24
becomes 24.0. With `aggfunc="sum"` you get a sum instead.

The values of `country` are now the row labels: with `.loc` you select rows by them, and
with `reset_index()` you turn them back into an ordinary column.

## Part 1 — World Bank population by country

The World Bank publishes the population of each country for every year since 1960. Each
number is an estimate of how many people live in the country in the middle of that year,
whatever their citizenship. The file
in `data/` is the CSV from the indicator's page,
[data.worldbank.org/indicator/SP.POP.TOTL](https://data.worldbank.org/indicator/SP.POP.TOTL),
as it downloads. It starts with four lines about the source before the header row, and with
`skiprows=4` you skip them.

In [ ]:
wb = pd.read_csv("data/world_bank_population.csv", skiprows=4)

wb.head()

**What is one row, and where are the years?**

**Which columns hold nothing useful?** Count the distinct values in each column that is not
a year.

**Which column says the same thing as another one?**

In [ ]:
# Sandbox

**Task 1.1: make `pop_long`, one row per country and year.**

- drop the columns that hold nothing useful, and the one that repeats another
- melt the years into one column, `year`, and their numbers into `population`
- make `year` a number
- drop the rows with no population

*You may need:* `drop(columns=...)`, `melt`, `astype(int)`, `dropna`.

In [ ]:
# Sandbox

In [ ]:
# Task 1.1 — pop_long: one row per country and year, with the columns Country Name,
# year (a number) and population. No row without a population.
pop_long = ...

check("pop_long", pop_long, "DataFrame")
check_col(pop_long, "year")
check_col(pop_long, "population")

**Is every row a country?** Sort the 2025 rows by population and read the top twenty
names.

In [ ]:
# Sandbox

**What would you need to count only the countries?**

A table for people to read is often wide, and you build one from `pop_long` now.

**Task 1.2: make `neighbours`, the population of Ukraine and six of its neighbours, with one row
per year and one column per country.**

*You may need:* `pivot_table`, `.loc`.

In [ ]:
# Sandbox

In [ ]:
# Task 1.2 — neighbours: from pop_long, the rows of these countries and years,
# with one row per year and one column per country.
countries = ["Ukraine", "Poland", "Slovak Republic", "Hungary", "Romania", "Moldova", "Belarus"]
years = [1990, 2000, 2010, 2020, 2025]
neighbours = ...

check("neighbours", neighbours, "DataFrame")

### Explore and discuss

**A chart, optional.** If you worked through the charts notebook this week, draw the
population of the seven countries for every year from 1990, one line per country. Which
table do you draw it from, `pop_long` or `neighbours`, and why?

**Discuss:** Ukraine's number for 2025 is lower than for 2020 by more than five million.
Before you report that fall, what would you check about how the number was estimated?
The indicator's page names its sources.

In [ ]:
# Explore

## Part 2 — Eurostat unemployment by age and sex

Eurostat publishes the unemployment rate of each European country per year, by age and sex.
The file in `data/` is the dataset `une_rt_a` as the Eurostat database sends it in TSV
format: values separated by tabs rather than commas. With `sep="\t"` you tell `read_csv` so.

In [ ]:
une = pd.read_csv("data/une_rt_a.tsv", sep="\t")

une.head()

**Read the name of the first column and the first values under it. How many variables does
this one column hold?**

**What do the values in the year columns look like?** Read a few rows and list what you find
besides numbers.

In [ ]:
# Sandbox

The name of the first column holds a backslash, which is awkward to type in Python code, so
we rename it to `codes`.

In [ ]:
une = une.rename(columns={une.columns[0]: "codes"})

Each value of `codes` has five parts separated by commas, in this order: `freq`, `age` (the age group, `Y15-74` is 15 to 74), `unit`, `sex`
(`F`, `M`, and `T` for total) and `geo` (Eurostat's country code, for example `EL` for
Greece).

**Task 2.1: make `une_long`, one row per code combination and year.**

- melt the years into one column, `year`, and the cells into `value`
- split `codes` into five columns: with `une_long[["freq", "age", "unit", "sex", "geo"]] = ...`
  you add all five in one step
- make `year` a number

*You may need:* `melt`, `.str.split(",", expand=True)`, `astype(int)`.

In [ ]:
# Sandbox

In [ ]:
# Task 2.1 — une_long: one row per code combination and year, with the columns freq,
# age, unit, sex, geo, year (a number) and value (the text of the cell, as in the file).
une_long = ...

check("une_long", une_long, "DataFrame")
check_col(une_long, "geo")
check_col(une_long, "year")

**Which of the five new columns holds only one value?**

**List the values of `age`. Which groups overlap, and what happens to a sum over all ages?**

In [ ]:
# Sandbox

The `value` column still holds text.

**Turn `value` into numbers with `pd.to_numeric`.** If it stops with an error, read the
error. Then run it again with `errors="coerce"`: with it you get a missing value in place of
each text that is not a number, instead of an error.

**How many values are missing now, and which texts became missing?** Compare it with how
many cells hold `:`. Look at exactly what those cells hold before you count.

In [ ]:
# Sandbox

The letters after some numbers are **flags**, notes from Eurostat about the value: `b` a
break in the series (the way of measuring changed that year), `d` a definition that differs,
`u` low reliability. A value can carry two flags at once: `bu` is `b` and `u`.

**Task 2.2: split `value` into two columns, `number` for the number and `flag` for the
letters.**

*You may need:* `.str.split(expand=True)`, `pd.to_numeric`.

In [ ]:
# Sandbox

In [ ]:
# Task 2.2 — two columns on une_long: number, the number in value as a float (missing
# where there is none), and flag, the letters after it (missing where there are none).
# une_long["number"] = ...
# une_long["flag"] = ...

check_col(une_long, "number")
check_col(une_long, "flag")

**Can you compare any two values in `number`?** Count the rows per `unit`, and read the
[dataset's page](https://ec.europa.eu/eurostat/databrowser/view/une_rt_a/default/table)
for what the three units are.

In [ ]:
# Sandbox

**Task 2.3: make `by_sex`, the rates of women and men side by side, and
`widest_gap_2025`, the country with the widest gap in 2025.**

- keep one unit, **`PC_ACT`**: the unemployed as a percentage of the labour force, which is
  everyone who has a job or is looking for one
- keep one age group, **`Y15-74`**
- put the three values of `sex` side by side, and add `gap`, women minus men

*You may need:* `pivot_table` with two columns in `index`, `.loc`, `idxmax`.

In [ ]:
# Sandbox

In [ ]:
# Task 2.3 — by_sex: the rows of unit PC_ACT and age Y15-74, one row per year and geo,
# with the columns F, M and T, and a column gap: F minus M.
# Then widest_gap_2025: the geo code with the largest gap in 2025.
by_sex = ...
# by_sex["gap"] = ...
widest_gap_2025 = ...

check("by_sex", by_sex, "DataFrame")
check_col(by_sex, "gap")
check("widest_gap_2025", widest_gap_2025, "text")

### Explore and discuss

**Countries and groups of countries.** List the values of `geo`. Which of them are not a
country?

**Is `T` the mean of `F` and `M`?** Compare `T` with the mean of the two, row by row. When
is it closer to `F`, and when to `M`? What would decide it?

**Flags.** Count the flags per country. Would you report a rate that carries `u`?

**Discuss:** in which countries is unemployment higher for women, and in which for men?
Before you compare two countries, which flags would you look at?

In [ ]:
# Explore

## Part 3 — The note

**A colleague asks: how many young people, aged 15 to 24, were unemployed in the EU in
2025?**

Find the answer in `une_long`. Then write it in the cell below, with one line for each choice
you made on the way to it, and run the cell.

In [ ]:
# Sandbox

> **The answer:** *(one number, and what it counts)*
>
> **The choices:** *(one line for each column you filtered on: the value you kept, and why)*

## Before you submit

**Say what you used.** One line naming the tool and the step it helped with, or
"No AI used".

> *Example: "Used Claude to see why astype(float) stopped on one value."*

In [ ]:
ai_disclosure = "..."

check("ai_disclosure", ai_disclosure, "text")

**Submit:** restart the kernel and run every cell from the top, check that every cell runs,
save, commit and push. Then paste your repository URL into the Week 5 practice assignment
on Moodle.

The deadline is shown on the Moodle assignment.